# 🌐 Multilingual Medical Query Translation (VI -> ZH) using Qwen2.5-7B-Instruct
Phục vụ cho cuộc thi R2AI Stage 3 - 2026. Tuân thủ 100% quy định BTC (Model <= 15B, open-source trên HF).
Chạy trên Kaggle GPU (T4 x 2 hoặc P100) mất ~3-5 phút cho toàn bộ 1,200 câu hỏi y khoa.

In [ ]:
!pip install -q transformers accelerate
import os, time, torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 1. Load queries
# Thay đường dẫn nếu bạn upload dataset query.parquet lên Kaggle
QUERY_PATH = 'query.parquet'
if not os.path.exists(QUERY_PATH):
    QUERY_PATH = '/kaggle/input/r2ai-queries/query.parquet'

df = pd.read_parquet(QUERY_PATH)
print(f'Total queries to translate: {len(df)}')

In [ ]:
# 2. Load Qwen2.5-7B-Instruct
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
print(f'Loading {MODEL_ID}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto'
)
model.eval()
print('Model loaded successfully!')

In [ ]:
# 3. Batch translation
SYSTEM_PROMPT = (
    'You are an expert medical translator. Translate the given Vietnamese patient inquiry '
    'or clinical query into Simplified Chinese accurately. Maintain all symptoms, anatomical terms, '
    'drug names, and clinical conditions. Output ONLY the translated Chinese text without explanations, '
    'notes, or commentary.'
)

BATCH_SIZE = 16
translated_texts = []
t0 = time.time()

for i in range(0, len(df), BATCH_SIZE):
    batch = df.iloc[i : i + BATCH_SIZE]
    batch_prompts = []
    for _, row in batch.iterrows():
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': row['query']}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        batch_prompts.append(text)
        
    inputs = tokenizer(batch_prompts, return_tensors='pt', padding=True, truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        
    input_len = inputs['input_ids'].shape[1]
    gen = outputs[:, input_len:]
    decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
    translated_texts.extend([d.strip() for d in decoded])
    
    done = min(i + BATCH_SIZE, len(df))
    speed = done / max(time.time() - t0, 0.001)
    print(f'Done: {done}/{len(df)} ({done/len(df)*100:.1f}%) | Speed: {speed:.1f} q/s')

df['query_zh'] = translated_texts
df.to_parquet('query_translated.parquet', index=False)
print('Saved query_translated.parquet!')

In [ ]:
# 4. Quick inspection
for i in range(min(5, len(df))):
    print(f'--- Query {i+1} ---')
    print('VI:', df.iloc[i]['query'][:150])
    print('ZH:', df.iloc[i]['query_zh'][:150])